# 02 - Prompt EngineeringTest and refine LLM prompts for claims extraction and classification.## SetupConfigure the GoldenRay MaaS client.

In [ ]:
import httpximport jsonimport os# Configuration — change these to match your GoldenRay setupAPI_URL = os.getenv('GOLDENRAY_API_URL', 'http://10.0.0.113:30888')NAMESPACE = os.getenv('GOLDENRAY_NAMESPACE', 'demo-tenant')ENDPOINT = os.getenv('GOLDENRAY_ENDPOINT', 'TinyLlama')API_KEY = os.getenv('GOLDENRAY_API_KEY', '')def call_llm(messages, max_tokens=300, temperature=0.1):    """Call GoldenRay MaaS inference API."""    url = f'{API_URL}/api/v1/inference/{NAMESPACE}/{ENDPOINT}'    headers = {'Content-Type': 'application/json'}    if API_KEY:        headers['Authorization'] = f'Bearer {API_KEY}'    resp = httpx.post(url, json={'messages': messages, 'max_tokens': max_tokens, 'temperature': temperature}, headers=headers, timeout=60)    data = resp.json()    if 'choices' in data:        return data['choices'][0]['message']['content']    return str(data)print(f'Configured: {API_URL}/api/v1/inference/{NAMESPACE}/{ENDPOINT}')

## Test Extraction Prompt

In [ ]:
from pathlib import Pathclaim_text = Path('../data/sample_claims/claim1.txt').read_text()system_prompt = '''You are an insurance claims processing assistant. Extract structured information from claim documents.Respond in valid JSON with: claim_id, claimant_name, incident_date, claim_amount, claim_type, description.'''messages = [    {'role': 'system', 'content': system_prompt},    {'role': 'user', 'content': f'Extract claim info from:\n\n{claim_text}\n\nRespond with JSON.'}]try:    result = call_llm(messages)    print('LLM Response:')    print(result)except Exception as e:    print(f'LLM call failed (expected if not connected to platform): {e}')    print('\nThis notebook requires access to the GoldenRay MaaS API.')

## Test Classification Prompt

In [ ]:
classification_system = '''You are a claims severity classifier.Rules: LOW (<$5k), MEDIUM ($5k-$25k), HIGH ($25k-$100k), CRITICAL (>$100k or injury/death).Respond in JSON: severity, confidence, reasoning.'''messages = [    {'role': 'system', 'content': classification_system},    {'role': 'user', 'content': 'Classify: auto claim, $15,500, highway collision with whiplash injury.'}]try:    result = call_llm(messages, max_tokens=200)    print('Classification:', result)except Exception as e:    print(f'Expected with small models — classification may need refinement: {e}')

## Prompt Iteration Tips**With small models (TinyLlama, opt-125m):**- Keep prompts short and explicit- Use few-shot examples- Expect partial/imperfect JSON — add robust parsing- Rule-based fallbacks are essential**With larger models (Llama-3-8B+, GPT-4o):**- Can handle complex multi-field extraction- Better at following JSON schema instructions- Can do zero-shot classification reliably